<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/Aula_01_Professor_Intro_Deep_Learning_IA_Generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 1 – Introdução ao Deep Learning e IA Generativa

**Curso:** Deep Learning e Inteligência Artificial Generativa Aplicada  
**Carga da aula:** 4 horas  
**Estratégia:** aula expositiva dialogada com demonstração prática em Google Colab.

## Capacidades trabalhadas
- Construir redes neurais artificiais.
- Demonstrar atenção a detalhes.

## Conhecimentos da aula
- Conceito de Inteligência Artificial.
- Machine Learning e Deep Learning.
- Diferença entre redes rasas e redes profundas.
- Aplicações industriais de Deep Learning.

## Evidência de aprendizagem
Ao final, o aluno deverá entregar um notebook com: leitura do dataset, análise inicial, comparação entre um modelo raso e uma rede neural simples, além de respostas técnicas sobre IA, ML, DL e aplicações industriais.

> **Versão do professor:** contém uma proposta de solução completa, comentários técnicos e respostas esperadas.

## 1. Contextualização industrial

Nesta aula, será usado um dataset fictício de monitoramento de máquinas industriais. Cada linha representa uma leitura operacional de uma máquina. O objetivo didático é classificar a operação como **normal** ou **risco_falha** com base em variáveis como temperatura, vibração, corrente, tempo de ciclo e pressão.

Esse exemplo permite discutir a diferença entre:

- **IA:** área ampla que busca criar sistemas capazes de executar tarefas associadas à inteligência.
- **Machine Learning:** subárea da IA em que modelos aprendem padrões a partir de dados.
- **Deep Learning:** subárea de ML que utiliza redes neurais com múltiplas camadas para aprender representações mais complexas.


In [ ]:
# Célula 1 - Importação das bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('Bibliotecas carregadas com sucesso!')
print('TensorFlow:', tf.__version__)

## 2. Carregamento do dataset

No Google Colab, envie o arquivo `dataset_aula1_intro_deep_learning_industrial.csv` para o ambiente ou mantenha-o na mesma pasta do notebook.

In [ ]:
# Célula 2 - Leitura do dataset
arquivo = 'dataset_aula1_intro_deep_learning_industrial.csv'
df = pd.read_csv(arquivo)

print('Dimensão do dataset:', df.shape)
df.head()

In [ ]:
# Célula 3 - Verificação geral dos dados
display(df.info())
display(df.describe(include='all'))
print('Valores ausentes por coluna:')
print(df.isna().sum())
print('Distribuição da classe alvo:')
print(df['classe_operacao'].value_counts())

## 3. Análise exploratória inicial

Antes de treinar qualquer modelo, é importante observar os dados. Isso ajuda a identificar inconsistências, valores ausentes, variáveis mais relevantes e equilíbrio das classes.

In [ ]:
# Célula 4 - Visualização da distribuição da variável alvo
df['classe_operacao'].value_counts().plot(kind='bar')
plt.title('Distribuição da classe de operação')
plt.xlabel('Classe')
plt.ylabel('Quantidade de amostras')
plt.show()

In [ ]:
# Célula 5 - Comparação de variáveis numéricas por classe
variaveis_numericas = ['temperatura_motor_c', 'vibracao_mm_s', 'corrente_a', 'tempo_ciclo_s', 'pressao_bar']

for coluna in variaveis_numericas:
    df.boxplot(column=coluna, by='classe_operacao')
    plt.title(f'{coluna} por classe')
    plt.suptitle('')
    plt.xlabel('Classe')
    plt.ylabel(coluna)
    plt.show()

## 4. Preparação dos dados

A rede neural precisa receber dados numéricos tratados. Nesta etapa vamos:

1. Separar variáveis de entrada e alvo.
2. Tratar valores ausentes.
3. Padronizar variáveis numéricas.
4. Codificar variáveis categóricas.
5. Separar treino e teste.

In [ ]:
# Célula 6 - Separação entre X e y
X = df.drop(columns=['id_amostra', 'classe_operacao'])
y = df['classe_operacao'].map({'normal': 0, 'risco_falha': 1})

colunas_numericas = ['temperatura_motor_c', 'vibracao_mm_s', 'corrente_a', 'tempo_ciclo_s', 'pressao_bar']
colunas_categoricas = ['maquina', 'turno']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)

In [ ]:
# Célula 7 - Pipeline de pré-processamento
preprocessador = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), colunas_numericas),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), colunas_categoricas)
    ]
)

X_train_prep = preprocessador.fit_transform(X_train)
X_test_prep = preprocessador.transform(X_test)

print('Formato após pré-processamento:', X_train_prep.shape)

## 5. Modelo raso: Machine Learning clássico

Para comparar com Deep Learning, vamos treinar um modelo mais simples, a regressão logística. Ele servirá como referência inicial.

In [ ]:
# Célula 8 - Treinamento de um modelo raso
modelo_raso = LogisticRegression(max_iter=1000, random_state=42)
modelo_raso.fit(X_train_prep, y_train)

y_pred_raso = modelo_raso.predict(X_test_prep)

print('Acurácia do modelo raso:', accuracy_score(y_test, y_pred_raso))
print('Relatório de classificação:')
print(classification_report(y_test, y_pred_raso, target_names=['normal', 'risco_falha']))

## 6. Rede neural artificial inicial

Agora criaremos uma rede neural com camadas densas. Esta estrutura já representa uma introdução ao Deep Learning, pois o modelo aprende representações internas a partir dos dados.

In [ ]:
# Célula 9 - Construção da rede neural
tf.random.set_seed(42)

n_entradas = X_train_prep.shape[1]

modelo_dl = Sequential([
    Dense(32, activation='relu', input_shape=(n_entradas,)),
    Dropout(0.15),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

modelo_dl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_dl.summary()

In [ ]:
# Célula 10 - Treinamento da rede neural
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

historico = modelo_dl.fit(
    X_train_prep, y_train,
    validation_split=0.2,
    epochs=80,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Célula 11 - Curvas de treinamento
plt.plot(historico.history['loss'], label='loss treino')
plt.plot(historico.history['val_loss'], label='loss validação')
plt.title('Evolução da função de perda')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.plot(historico.history['accuracy'], label='accuracy treino')
plt.plot(historico.history['val_accuracy'], label='accuracy validação')
plt.title('Evolução da acurácia')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.show()

In [ ]:
# Célula 12 - Avaliação da rede neural
probabilidades = modelo_dl.predict(X_test_prep).ravel()
y_pred_dl = (probabilidades >= 0.5).astype(int)

print('Acurácia da rede neural:', accuracy_score(y_test, y_pred_dl))
print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred_dl))
print('Relatório de classificação:')
print(classification_report(y_test, y_pred_dl, target_names=['normal', 'risco_falha']))

## 7. Comparação: rede rasa x rede profunda

**Resposta esperada do professor:**

- O modelo raso pode funcionar bem em problemas com relações mais simples entre as variáveis.
- A rede neural pode aprender relações não lineares mais complexas, principalmente quando há grande volume de dados e padrões difíceis de representar manualmente.
- Nem sempre Deep Learning será melhor que ML clássico em datasets pequenos ou simples.
- A escolha do modelo deve considerar objetivo, dados disponíveis, custo computacional, interpretabilidade e desempenho esperado.

In [ ]:
# Célula 13 - Comparação simples dos resultados
comparacao = pd.DataFrame({
    'Modelo': ['Regressão Logística - modelo raso', 'Rede Neural - Deep Learning inicial'],
    'Acurácia': [accuracy_score(y_test, y_pred_raso), accuracy_score(y_test, y_pred_dl)]
})

display(comparacao)
comparacao.plot(x='Modelo', y='Acurácia', kind='bar', legend=False)
plt.ylim(0, 1)
plt.title('Comparação de desempenho')
plt.ylabel('Acurácia')
plt.show()

## 8. Atividade de consolidação – Respostas esperadas

1. **O que é Inteligência Artificial?**  
Área da computação voltada ao desenvolvimento de sistemas capazes de executar tarefas associadas à inteligência, como reconhecimento de padrões, tomada de decisão e geração de respostas.

2. **Qual a diferença entre Machine Learning e Deep Learning?**  
Machine Learning usa algoritmos que aprendem padrões a partir dos dados. Deep Learning é uma subárea de ML baseada em redes neurais com múltiplas camadas, capaz de aprender representações complexas.

3. **O que diferencia uma rede rasa de uma rede profunda?**  
Redes rasas possuem poucas camadas intermediárias. Redes profundas possuem múltiplas camadas ocultas, permitindo maior capacidade de representação.

4. **Cite uma aplicação industrial de Deep Learning.**  
Inspeção visual de peças, manutenção preditiva, classificação de falhas, análise de imagens, processamento de chamados técnicos ou automação de atendimento.

5. **Por que é importante demonstrar atenção a detalhes no desenvolvimento de modelos de IA?**  
Porque pequenos erros em dados, configuração de camadas, métricas, divisão de treino/teste ou interpretação dos resultados podem comprometer o desempenho e a confiabilidade da solução.

## 9. Orientações para entrega no Google Classroom

O aluno deverá entregar:

- Notebook executado;
- Gráficos gerados;
- Comparação entre modelo raso e rede neural;
- Respostas técnicas da atividade de consolidação;
- Breve conclusão sobre a aplicação industrial desenvolvida.